In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path

print("torch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

In [ ]:
#pip install openpyxl

In [ ]:
# ── Settings ────────────────────────────────────────────────────────
DATA_DIR = Path("")

EXCLUDE_FILES = []  # Example of a file to exclude, add more if needed

FEATURE_COLS = [
    "E",
    "Norm",
    "Indentation depth",
    "Compression depth",
    "Force of compression",
    "Cantilever Deflection",
    "∆H"
]

METADATA_COLS = [
    "Treatment",
    "Line",
    "Moment",
    "Cell number",
    "Replicate",
    "Contact time",
    "Force of compression",
    "Compression Speed",
    "source_file",
]

# ── Load all files ───────────────────────────────────────────────────
files = [f for f in DATA_DIR.glob("*.xlsx") 
         if f.name not in EXCLUDE_FILES]

dfs = []
for f in files:
    print(f"Loading: {f.name}")
    df = pd.read_excel(f)
    df["source_file"] = f.name
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows: {len(df_all)}")

# ── Merge Ctrl into DMSO ─────────────────────────────────────────────
df_all["Treatment"] = df_all["Treatment"].replace({"Ctrl": "DMSO"})

# ── Extract features ─────────────────────────────────────────────────
X_raw = df_all[FEATURE_COLS].apply(pd.to_numeric, errors="coerce")
valid_mask = X_raw.notna().all(axis=1)
print(f"Dropping {(~valid_mask).sum()} rows with missing values")

X = X_raw[valid_mask].reset_index(drop=True)
meta = df_all[METADATA_COLS][valid_mask].reset_index(drop=True)

print(f"\nFinal dataset: {X.shape[0]} rows × {X.shape[1]} features")
print(f"Treatments: {meta['Treatment'].unique()}")
print(f"Files: {meta['source_file'].unique()}")

In [ ]:
print(pd.crosstab(meta["Treatment"], meta["Replicate"]))

In [ ]:
## create a conditional VAE, knowing which file each measurement comes from we can correct for it
## condition created by source files
# One-hot encode source file
files = meta["source_file"].values
unique_files = sorted(set(files))
file_to_idx = {f: i for i, f in enumerate(unique_files)}

print("File to index mapping:")
for f, i in file_to_idx.items():
    print(f"  {i}: {f}")

# Create one-hot matrix
n_files = len(unique_files)
C = np.zeros((len(meta), n_files))
for i, f in enumerate(files):
    C[i, file_to_idx[f]] = 1.0

# Convert to tensor
C_tensor = torch.FloatTensor(C)
print(f"\nOne-hot matrix shape: {C_tensor.shape}")
print(f"Example (first row): {C_tensor[0]}")

In [ ]:
## condition created by replicates
replicates = meta["Replicate"].astype(str).values
unique_replicates = sorted(set(replicates))
rep_to_idx = {r: i for i, r in enumerate(unique_replicates)}

n_reps = len(unique_replicates)
C = np.zeros((len(meta), n_reps))
for i, r in enumerate(replicates):
    C[i, rep_to_idx[r]] = 1.0

C_tensor = torch.FloatTensor(C)
print(f"Unique replicates: {unique_replicates}")
print(f"One-hot matrix shape: {C_tensor.shape}")

In [ ]:
meta["batch"] = meta["source_file"].str.replace(".xlsx", "") + "_R" + meta["Replicate"].astype(str)
print(f"Unique batches: {meta['batch'].nunique()}")
print(meta["batch"].value_counts().sort_index())

In [ ]:
## condition created by replicates
replicates = meta["batch"].astype(str).values
unique_replicates = sorted(set(replicates))
rep_to_idx = {r: i for i, r in enumerate(unique_replicates)}

n_reps = len(unique_replicates)
C = np.zeros((len(meta), n_reps))
for i, r in enumerate(replicates):
    C[i, rep_to_idx[r]] = 1.0

C_tensor = torch.FloatTensor(C)
print(f"Unique replicates: {unique_replicates}")
print(f"One-hot matrix shape: {C_tensor.shape}")

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert to tensor
X_tensor = torch.FloatTensor(X_scaled)
print(f"Tensor shape: {X_tensor.shape}")
print(f"Min: {X_tensor.min():.2f}, Max: {X_tensor.max():.2f}, Mean: {X_tensor.mean():.2f}")

In [ ]:
print(f"X_scaled rows: {X_scaled.shape[0]}")
print(f"C rows: {C.shape[0]}")

In [ ]:
# concatenate features + condition
X_combined = np.concatenate([X_scaled, C], axis=1)
X_combined_tensor = torch.FloatTensor(X_combined)
print(f"Combined input shape: {X_combined_tensor.shape}")
print("Feature columns:", X.columns.tolist())
print("Feature matrix shape:", X.shape)

In [ ]:
##Normal MechanoENV
class MechanoVAE(nn.Module):
    def __init__(self, input_dim=7, hidden_dim=4, latent_dim=2):
        super(MechanoVAE, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
        
        # Two heads
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_var = nn.Linear(hidden_dim, latent_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
    
    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        log_var = self.fc_var(h)
        return mu, log_var
    
    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        x_reconstructed = self.decode(z)
        return x_reconstructed, mu, log_var

In [ ]:
## Conditional MechanoENV
class CondMechanoVAE(nn.Module):
    def __init__(self, input_dim=7, condition_dim=7, hidden_dim=4, latent_dim=2):
        super(CondMechanoVAE, self).__init__()
        
        # Encoder takes features + condition
        self.encoder = nn.Sequential(
            nn.Linear(input_dim + condition_dim, hidden_dim),
            nn.ReLU()
        )
        
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_var = nn.Linear(hidden_dim, latent_dim)
        
        # Decoder takes z + condition
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim + condition_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
    
    def encode(self, x, c):
        xc = torch.cat([x, c], dim=1)
        h = self.encoder(xc)
        mu = self.fc_mu(h)
        log_var = self.fc_var(h)
        return mu, log_var
    
    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z, c):
        zc = torch.cat([z, c], dim=1)
        return self.decoder(zc)
    
    def forward(self, x, c):
        mu, log_var = self.encode(x, c)
        z = self.reparameterize(mu, log_var)
        x_reconstructed = self.decode(z, c)
        return x_reconstructed, mu, log_var
    
        """_summary_: we sum instead of multiplying the latent space with 
        the condition, this is a common approach in conditional VAEs and 
        allows the model to learn how to combine the latent representation 
        with the condition in a more flexible way.
        once we concatenated we got a vector of length 14 where at positions 0-6 mechanical features sit
        and at position 7-13 sits the file identity. The linear layer then does: output = [x,c x W +b]
        where W shape 14x4. This means each of the 4 hidden neurons is a weighted sum of all 14 inputs. the network learns by combiation.
        Also the conditions have many zeros and only one 1 hence, by multiplying we would have only one condition contributiog to the latent 
        space, while by summing we allow all conditions to contribute in a more balanced way.
        """

In [ ]:
device = torch.device("mps")
model = MechanoVAE(input_dim=7, hidden_dim=4, latent_dim=2).to(device)
print(model)
print(f"\nModel is on device: {next(model.parameters()).device}")

In [ ]:
device = torch.device("mps")
model = CondMechanoVAE(input_dim=7,condition_dim =26, hidden_dim=16, latent_dim=2).to(device)
print(model)
print(f"\nModel is on device: {next(model.parameters()).device}")

In [ ]:
def vae_loss(x_reconstructed, x_original, mu, log_var):
    reconstruction_loss = nn.functional.mse_loss(
        x_reconstructed, x_original, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return reconstruction_loss + kl_loss

In [ ]:
## normal VAE
dataset = TensorDataset(X_tensor)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 500
losses = []

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    for batch in dataloader:
        x = batch[0].to(device)
        x_reconstructed, mu, log_var = model(x)
        loss = vae_loss(x_reconstructed, x, mu, log_var)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    if epoch % 100 == 0:
        print(f"Epoch {epoch}/{n_epochs} — Loss: {avg_loss:.1f}")

In [ ]:
## conditional VAE
dataset = TensorDataset(X_tensor, C_tensor) # we need to pass both the features and the conditions to the dataloader
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 500
losses = []

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    for batch in dataloader:
        x = batch[0].to(device)
        c = batch[1].to(device)
        
        x_reconstructed, mu, log_var = model(x, c)
        loss = vae_loss(x_reconstructed, x, mu, log_var)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    if epoch % 100 == 0:
        print(f"Epoch {epoch}/{n_epochs} — Loss: {avg_loss:.1f}")
        """TensorDataset guarantees that when the dataloader shuffles the data, x and c are always shuffled together.
        """

In [ ]:
# Extract latent space — no UMAP needed, already 2D

### normal VAE
#model.eval()
#with torch.no_grad():
#    X_gpu = X_tensor.to(device)
#    mu, log_var = model.encode(X_gpu)
#    Z = mu.cpu().numpy()
 ## conditional VAE
model.eval()
with torch.no_grad():
    X_gpu = X_tensor.to(device)
    C_gpu = C_tensor.to(device)
    mu, log_var = model.encode(X_gpu, C_gpu)
    Z = mu.cpu().numpy()

print(f"Latent space shape: {Z.shape}")    

print(f"Latent space shape: {Z.shape}")

# Plot colored by treatment
fig, ax = plt.subplots(figsize=(8, 6))
treatments = meta["Treatment"].values
unique_treatments = sorted(set(treatments))
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_treatments)))

for treatment, color in zip(unique_treatments, colors):
    mask = treatments == treatment
    ax.scatter(Z[mask, 0], Z[mask, 1],
               label=treatment, color=color, alpha=0.6, s=15, edgecolors="none")

ax.set_xlabel("z1")
ax.set_ylabel("z2")
ax.set_title("MechanoVAE latent space — colored by treatment")
ax.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
files = meta["source_file"].values
unique_files = sorted(set(files))
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_files)))

for f, color in zip(unique_files, colors):
    mask = files == f
    ax.scatter(Z[mask, 0], Z[mask, 1],
               label=f, color=color, alpha=0.6, s=15, edgecolors="none")

ax.set_xlabel("z1")
ax.set_ylabel("z2")
ax.set_title("MechanoVAE latent space — colored by source file")
ax.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
"""I had a problem with IPA-3 and tried to find the source of it
Batch correction is not real here, i should correct for replicate and not for file, 
because the problem is that IPA-3 is only in one replicate, so if I correct for file I 
am correcting for the problem but if I correct for replicate I am correcting for the source of the problem."""

In [ ]:
# Cross-tabulate treatment vs source file
print(pd.crosstab(meta["Treatment"], meta["source_file"]))

In [ ]:
print(meta.columns.tolist())
print(meta["Replicate"].unique() if "Replicate" in meta.columns else "No replicate column")
print(pd.crosstab(meta["Treatment"], meta["Replicate"]))